In [1]:
# Imports
import pandas as pd
import numpy as np
import os

In [2]:
# Define paths
RAW_PATH = "../data/raw/"
PROCESSED_PATH = "../data/processed/"

In [4]:
# Load all CSV files 
files = {
    "sessions": "website_sessions.csv",
    "orders": "orders.csv",
    "order_items": "order_items.csv",
    "products": "products.csv",
    "refunds": "order_item_refunds.csv",   
    "pageviews": "website_pageviews.csv",  
    
}

data = {}
for name, filename in files.items():
    filepath = os.path.join(RAW_PATH, filename)
    if os.path.exists(filepath):
        data[name] = pd.read_csv(filepath)
        print(f"Loaded {name}: {data[name].shape}")
    else:
        print(f"File not found: {filename}")

Loaded sessions: (472871, 9)
Loaded orders: (32313, 8)
Loaded order_items: (40025, 7)
Loaded products: (4, 3)
Loaded refunds: (1731, 5)
Loaded pageviews: (1188124, 4)


In [5]:
# Quick inspection of each table
for name, df in data.items():
    print(f"\n=== {name.upper()} ===")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"Nulls:\n{df.isnull().sum()}")
    print(f"First 2 rows:\n{df.head(2)}")


=== SESSIONS ===
Shape: (472871, 9)
Columns: ['website_session_id', 'created_at', 'user_id', 'is_repeat_session', 'utm_source', 'utm_campaign', 'utm_content', 'device_type', 'http_referer']
Nulls:
website_session_id        0
created_at                0
user_id                   0
is_repeat_session         0
utm_source            83328
utm_campaign          83328
utm_content           83328
device_type               0
http_referer          39917
dtype: int64
First 2 rows:
   website_session_id           created_at  user_id  is_repeat_session  \
0                   1  2012-03-19 08:04:16        1                  0   
1                   2  2012-03-19 08:16:49        2                  0   

  utm_source utm_campaign utm_content device_type             http_referer  
0    gsearch     nonbrand      g_ad_1      mobile  https://www.gsearch.com  
1    gsearch     nonbrand      g_ad_1     desktop  https://www.gsearch.com  

=== ORDERS ===
Shape: (32313, 8)
Columns: ['order_id', 'created_at',

In [6]:
# Merge sessions with orders to create a session-level fact table
# Check actual column names first – common ones: website_session_id, order_id, price_usd
if "sessions" in data and "orders" in data:
    sessions = data["sessions"].copy()
    orders = data["orders"].copy()
    
    # Identify the correct key columns (adjust if needed after inspection)
    # Usually 'website_session_id' is the link
    # Orders table may have 'website_session_id' or 'session_id'
    
    # For now, assume both have 'website_session_id' and orders has 'order_id' and 'price_usd'
    fact_sessions = sessions.merge(
        orders[["website_session_id", "order_id", "price_usd"]], 
        on="website_session_id", 
        how="left"
    )
    
    # Create conversion flag
    fact_sessions["is_converted"] = np.where(fact_sessions["order_id"].notna(), 1, 0)
    
    # Fill missing revenue with 0
    fact_sessions["price_usd"] = fact_sessions["price_usd"].fillna(0)
    
    print(f"✅ fact_sessions created: {fact_sessions.shape}")
    print(f"Conversion rate: {fact_sessions['is_converted'].mean():.2%}")
    
    # Save
    fact_sessions.to_csv(os.path.join(PROCESSED_PATH, "fact_sessions.csv"), index=False)
    print("💾 Saved fact_sessions.csv")
else:
    print("⚠️ Missing sessions or orders table – cannot create fact table")

✅ fact_sessions created: (472871, 12)
Conversion rate: 6.83%
💾 Saved fact_sessions.csv


In [7]:
# Save individual cleaned tables 
for name, df in data.items():
    df.to_csv(os.path.join(PROCESSED_PATH, f"{name}_cleaned.csv"), index=False)
print("All tables saved to data/processed/")

All tables saved to data/processed/


In [8]:
# Quick check of website_pageviews – useful for funnel analysis
if "pageviews" in data:
    print("\n=== WEBSITE_PAGEVIEWS preview ===")
    print(data["pageviews"].head())
    print(f"Unique pages: {data['pageviews']['pageview_url'].nunique()}")
    print(f"Top pages:\n{data['pageviews']['pageview_url'].value_counts().head()}")


=== WEBSITE_PAGEVIEWS preview ===
   website_pageview_id           created_at  website_session_id pageview_url
0                    1  2012-03-19 08:04:16                   1        /home
1                    2  2012-03-19 08:16:49                   2        /home
2                    3  2012-03-19 08:26:55                   3        /home
3                    4  2012-03-19 08:37:33                   4        /home
4                    5  2012-03-19 09:00:55                   5        /home
Unique pages: 16
Top pages:
pageview_url
/products                 261231
/the-original-mr-fuzzy    162525
/home                     137576
/lander-2                 131170
/cart                      94953
Name: count, dtype: int64
